# 17-phase17-scale-up

**neuron Phase 17** — paradigm 의 dynamic method (DST, Net2Net grow) 의 **advantage 가 scale-up 으로 발현되는지** 검증.

**배경**: Phase 16a/16b 둘 다 framework 검증은 PASS 했으나 *expected performance benefit 미실현*. Phase 16b Notion 분석에서 원인을 *본 setup (~1M params, 1500 step) 의 scale 한계* 로 결론. RigL/Net2Net 의 reference paper 들이 large-scale 에서만 advantage 발현하는 알려진 패턴.

**핵심 가설** (scale-up 효과):
1. **DST_RigL ≤ static_prune (16a 의 advantage 발현)** — 더 긴 학습으로 swap cost 가 amortized?
2. **grown ≈ large_baseline (16b 의 gap 축소)** — 더 긴 post-grow 학습으로 새 capacity 활성화?
3. **scale 자체의 효과 분리** — control mode 로 hyperparameter 와 scale 효과 분리 (Copilot #3315013786 반영)
4. **all-finite + smooth convergence** — 확대된 setup 에서도 안정성?

설계: 5 mode × 2 seed = 10 run at **scale-up config**.

| 항목 | Phase 16 | Phase 17 (scale-up) | 배수 |
|---|---|---|---|
| hidden_dim | 128 | **256** | 2x |
| n_layers | 4 | **6** | 1.5x |
| ffn_dim (large) | 256 | **512** | 2x |
| max_steps | 1500 | **5000** | 3.3x |
| approx params | ~1M | **~5M** | ~5x |

5 mode:
- `dense_large`: 큰 모델 baseline (target reference)
- `static_prune_50`: dense 학습 후 step 2500 에서 50% prune (Phase 15 at scale)
- `DST_RigL_p16hp`: **Phase 16a 동일 hyperparameter** (period=50, swap=10%) at Phase 17 scale → **scale 효과 only 분리**
- `DST_RigL_p17hp`: Phase 17 의 보수적 hyperparameter (period=200, swap=5%) at Phase 17 scale → **scale + hp 동시 효과**
- `grown`: ffn=256 시작 → step 2500 에서 512 로 grow (Phase 16b at scale)

**Confound 명시 (Copilot #3315013786)**: `DST_RigL_p17hp` 만 보면 *scale 변화* 와 *hyperparameter 변화* 가 confounded. 두 효과 분리를 위해 `DST_RigL_p16hp` (Phase 16a 와 동일 hp) 를 control 로 추가 → Phase 16a 결과와 직접 비교 시 *순수 scale 효과* 측정 가능.

arch 고정: `hybrid_around_one_around_one` + `use_full_graph=True`
데이터: TinyShakespeare (char-LM, block_size=64 유지)
시드: [42, 123]
작성일: 2026-05-28
연관: Issue [#80](https://github.com/EinSofINTEREST/GraphLM/issues/80) / Phase 16 메인 [#75](https://github.com/EinSofINTEREST/GraphLM/issues/75)

## 0. 환경 / 의존성

In [ ]:
from __future__ import annotations

import math
import statistics
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    load_tinyshakespeare_text,
)
from graphlm.neuron.hybrid_transformer_demo import (
    HybridTransformerTrainConfig,
    train_hybrid_transformer_lm,
)
from graphlm.utils import safe_perplexity

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")

## 1. Config — scale-up

In [ ]:
text = load_tinyshakespeare_text()
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
vocab_size = tokenizer.vocab_size
print(f"vocab_size = {vocab_size}, dataset size = {len(dataset)}")

# scale-up — Phase 16 대비 hidden 2x, layers 1.5x, ffn 2x, steps 3.3x
HIDDEN_DIM = 256
N_HEADS = 8  # head_dim = 32 (256/8)
N_LAYERS = 6
GROUP_SIZE = 16
BLOCK_SIZE = 64
BATCH_SIZE = 32
LR = 3e-4
MAX_STEPS = 5000
DYNAMIC_AT_STEP = MAX_STEPS // 2  # 2500 — prune/grow/DST 시작 시점
# Phase 16a hyperparameter (control — scale 효과 only 분리)
DST_PERIOD_P16 = 50
DST_SWAP_FRACTION_P16 = 0.1
# Phase 17 hyperparameter (보수적 — 추가로 swap-cost 완화 시도)
DST_PERIOD_P17 = 200
DST_SWAP_FRACTION_P17 = 0.05
# 빠른 테스트 (MAX_STEPS<500) 대응: max(1, ...) 로 음수 방지 (gemini #3314998767)
DST_END_STEP = max(1, MAX_STEPS - 500)  # 마지막 500 step stabilize
SEEDS = [42, 123]
ARCH = "hybrid_around_one_around_one"
FFN_SMALL = 256
FFN_LARGE = 512

# 5 mode 정의 — Copilot #3315013786 confound 분리 위해 DST 두 hp variant 모두 포함
MODES = [
    {
        "name": "dense_large",
        "ffn_dim": FFN_LARGE,
        "prune_at_step": None,
        "prune_fraction": 0.0,
        "regrow_method": None,
        "dst_period": None,
        "dst_swap_fraction": 0.0,
        "grow_at_step": None,
        "grow_ffn_target": None,
    },
    {
        "name": "static_prune_50",
        "ffn_dim": FFN_LARGE,
        "prune_at_step": DYNAMIC_AT_STEP,
        "prune_fraction": 0.5,
        "regrow_method": None,
        "dst_period": None,
        "dst_swap_fraction": 0.0,
        "grow_at_step": None,
        "grow_ffn_target": None,
    },
    {
        "name": "DST_RigL_p16hp",
        "ffn_dim": FFN_LARGE,
        "prune_at_step": DYNAMIC_AT_STEP,
        "prune_fraction": 0.5,
        "regrow_method": "rigl",
        "dst_period": DST_PERIOD_P16,  # Phase 16a 와 동일
        "dst_swap_fraction": DST_SWAP_FRACTION_P16,
        "grow_at_step": None,
        "grow_ffn_target": None,
    },
    {
        "name": "DST_RigL_p17hp",
        "ffn_dim": FFN_LARGE,
        "prune_at_step": DYNAMIC_AT_STEP,
        "prune_fraction": 0.5,
        "regrow_method": "rigl",
        "dst_period": DST_PERIOD_P17,  # Phase 17 보수적
        "dst_swap_fraction": DST_SWAP_FRACTION_P17,
        "grow_at_step": None,
        "grow_ffn_target": None,
    },
    {
        "name": "grown",
        "ffn_dim": FFN_SMALL,
        "prune_at_step": None,
        "prune_fraction": 0.0,
        "regrow_method": None,
        "dst_period": None,
        "dst_swap_fraction": 0.0,
        "grow_at_step": DYNAMIC_AT_STEP,
        "grow_ffn_target": FFN_LARGE,
    },
]
print("\nScale-up config:")
print(
    f"  hidden={HIDDEN_DIM}, n_heads={N_HEADS}, n_layers={N_LAYERS}, ffn(L)={FFN_LARGE}, group={GROUP_SIZE}"
)
print(f"  max_steps={MAX_STEPS}, dynamic_at={DYNAMIC_AT_STEP}")
print(f"  DST p16hp: period={DST_PERIOD_P16}, swap={DST_SWAP_FRACTION_P16} (Phase 16a 동일)")
print(f"  DST p17hp: period={DST_PERIOD_P17}, swap={DST_SWAP_FRACTION_P17} (보수적)")

## 2. Sweep 실행 (5 mode × 2 seed = 10 run, GPU 권장)

In [ ]:
results = {}
for mode in MODES:
    for seed in SEEDS:
        key = (mode["name"], seed)
        print(f"\n== mode={mode['name']} seed={seed} ==")
        cfg = HybridTransformerTrainConfig(
            dataset=dataset,
            vocab_size=vocab_size,
            hidden_dim=HIDDEN_DIM,
            n_heads=N_HEADS,
            ffn_dim=mode["ffn_dim"],
            n_layers=N_LAYERS,
            group_size=GROUP_SIZE,
            arch=ARCH,
            use_full_graph=True,
            block_size=BLOCK_SIZE,
            batch_size=BATCH_SIZE,
            lr=LR,
            max_steps=MAX_STEPS,
            prune_at_step=mode["prune_at_step"],
            prune_fraction=mode["prune_fraction"],
            regrow_method=mode["regrow_method"],
            dst_period=mode["dst_period"],
            dst_swap_fraction=mode["dst_swap_fraction"],
            dst_end_step=DST_END_STEP if mode["regrow_method"] else None,
            grow_at_step=mode["grow_at_step"],
            grow_ffn_target=mode["grow_ffn_target"],
            seed=seed,
            device=device,
        )
        out = train_hybrid_transformer_lm(cfg)
        results[key] = out
        n_cycles = len(out["dst_cycles"])
        print(
            f"  final_loss = {out['final_loss']:.4f}  (ppl = {safe_perplexity(out['final_loss']):.2f})"
            f"  params = {out['final_param_count']:,}"
            f"  sparsity = {out['final_sparsity']:.3f}  dst_cycles = {n_cycles}"
        )

## 3. 결과 표 + 자동 verdict

In [ ]:
print(
    f"{'mode':>16s} {'seed':>6s} {'final_loss':>12s} {'perplexity':>12s} "
    f"{'params':>10s} {'sparsity':>10s}"
)
print("-" * 75)
for (name, seed), out in results.items():
    fl = out["final_loss"]
    print(
        f"{name:>16s} {seed:>6d} {fl:>12.4f} {safe_perplexity(fl):>12.2f} "
        f"{out['final_param_count']:>10,} {out['final_sparsity']:>10.3f}"
    )

# mode 별 평균
print("\n== Mode summary (mean ± σ across seeds) ==")
summary = {}
for mode in MODES:
    name = mode["name"]
    vals = [results[(name, s)]["final_loss"] for s in SEEDS]
    params = [results[(name, s)]["final_param_count"] for s in SEEDS]
    sparsities = [results[(name, s)]["final_sparsity"] for s in SEEDS]
    m = statistics.mean(vals)
    sd = statistics.stdev(vals) if len(vals) > 1 else 0.0
    summary[name] = (m, sd, statistics.mean(params), statistics.mean(sparsities))
    print(
        f"  {name:>16s} {m:.4f} ± {sd:.4f}  (ppl ≈ {safe_perplexity(m):.2f})  "
        f"params={statistics.mean(params):,.0f}  sparsity={statistics.mean(sparsities):.3f}"
    )

# 자동 verdict
print("\n== Verdict ==")
dense_loss = summary["dense_large"][0]
static_loss = summary["static_prune_50"][0]
rigl_p16_loss = summary["DST_RigL_p16hp"][0]
rigl_p17_loss = summary["DST_RigL_p17hp"][0]
grown_loss = summary["grown"][0]

# 1. all-finite
all_finite = all(math.isfinite(out["final_loss"]) for out in results.values())
verdict_1 = "PASS" if all_finite else "FAIL"
print(f"1. all-finite: {all_finite}  [{verdict_1}]")

# 2. Phase 16a 가설 — 순수 scale 효과 (p16hp control) 로 검증
diff_rigl_static = rigl_p16_loss - static_loss
verdict_2 = "PASS" if diff_rigl_static <= 0.02 else "FAIL"
print(
    f"2. DST_RigL_p16hp ≤ static + 0.02 (순수 scale 효과): diff = {diff_rigl_static:+.4f}  [{verdict_2}]"
)
if diff_rigl_static < 0:
    print("   → DST advantage 발현! Phase 16a 의 scale-limit 가설 입증 (순수 scale 효과).")
elif diff_rigl_static < 0.005:
    print("   → DST 가 static 근방까지 도달 — 부분 advantage.")
else:
    print("   → DST 여전히 static 보다 약간 열위 — 추가 scale-up 필요.")

# 2b. hyperparameter 효과 — p17hp vs p16hp (같은 scale 에서 hp 만 다름)
diff_hp = rigl_p17_loss - rigl_p16_loss
print(f"   2b. hp 효과: p17hp - p16hp = {diff_hp:+.4f}  (음수면 보수적 hp 가 도움)")

# 3. Phase 16b 가설 재검증 — grown ≤ dense_large + 0.02
diff_grown_dense = grown_loss - dense_loss
verdict_3 = "PASS" if diff_grown_dense <= 0.02 else "FAIL"
print(f"3. grown ≤ dense_large + 0.02 (16b 재검증): diff = {diff_grown_dense:+.4f}  [{verdict_3}]")
if diff_grown_dense < 0.01:
    print("   → grown 이 dense_large 근방 — 0-init activation 충분.")
else:
    print("   → grown 과 dense 격차 여전 — post-grow 학습 시간 더 필요할 수 있음.")

# 4. plateau 도달 확인 — 마지막 500 step 의 loss 변동 < threshold
# 짧은 학습 (MAX_STEPS<500) 대응: 동적 half 분할 (gemini #3314998775)
print("\n   plateau 분석 (마지막 500 step rolling mean 변동):")
for name in [m["name"] for m in MODES]:
    for seed in SEEDS:
        losses = results[(name, seed)]["losses"]
        last_500 = losses[-500:]
        half_len = len(last_500) // 2
        if half_len > 0:
            first_half = sum(last_500[:half_len]) / half_len
            second_half = sum(last_500[half_len:]) / (len(last_500) - half_len)
            plateau_drift = second_half - first_half
        else:
            plateau_drift = 0.0
        print(
            f"     {name:>16s} seed={seed}: drift = {plateau_drift:+.4f}  (< -0.01 = still learning)"
        )

## 4. Loss curve 시각화

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
colors = {
    "dense_large": "tab:green",
    "static_prune_50": "tab:red",
    "DST_RigL_p16hp": "tab:purple",
    "DST_RigL_p17hp": "tab:blue",
    "grown": "tab:orange",
}
window = 100  # scale-up 이라 더 긴 rolling window

for mode in MODES:
    name = mode["name"]
    losses_per_seed = [results[(name, s)]["losses"] for s in SEEDS]
    smoothed = []
    for losses in losses_per_seed:
        smoothed.append(
            [
                sum(losses[max(0, i - window + 1) : i + 1]) / min(i + 1, window)
                for i in range(len(losses))
            ]
        )
    arr = torch.tensor(smoothed)
    mean = arr.mean(dim=0)
    # single seed 시 std=NaN 회피 (gemini #3314998777)
    std = arr.std(dim=0) if arr.size(0) > 1 else torch.zeros_like(mean)
    steps = list(range(len(mean)))
    ax.plot(steps, mean, label=name, color=colors[name], linewidth=1.5)
    ax.fill_between(steps, mean - std, mean + std, color=colors[name], alpha=0.12)

# dynamic step + DST end 수직선
ax.axvline(
    DYNAMIC_AT_STEP,
    color="black",
    linestyle=":",
    alpha=0.5,
    label=f"prune/grow @ {DYNAMIC_AT_STEP}",
)
ax.axvline(DST_END_STEP, color="gray", linestyle=":", alpha=0.4, label=f"DST end @ {DST_END_STEP}")

ax.set_xlabel("step")
ax.set_ylabel(f"loss (rolling mean w={window})")
ax.set_title(
    "Phase 17 scale-up (hidden=256, n_layers=6, ffn(L)=512, steps=5000) — 5 mode (mean ± σ over 2 seeds)"
)
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()

out_dir = Path("../../runs/notebook-neuron-phase17")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {out_dir / 'loss_curves.png'}")

## 5. Phase 15/16/17 합산 비교 (scale 영향 시각화)

In [ ]:
# Phase 16 baseline 수치 — Notion 정리 페이지 출처 (Copilot #3315013726 — 향후 CSV 자동 로드 검토)
# 출처:
#   Phase 16a: https://www.notion.so/36de8b70b7aa819aaa54d561fc222fa1
#   Phase 16b: https://www.notion.so/36ee8b70b7aa811bb354c4fc98a484ed
# Config: hidden=128, n_layers=4, ffn=256 (16a) / 256↔128→256 (16b), 1500 step, seeds [42, 123]
p16a_baselines = {
    "dense": 1.8999,  # Phase 16a dense (ffn=256)
    "static_50": 1.9777,  # Phase 16a static_50
    "DST_RigL_50": 1.9831,  # Phase 16a DST_RigL_50 (period=50, swap=0.1) — Phase 17 p16hp 와 직접 비교
}
p16b_grown = 1.9428  # Phase 16b grown (ffn 128 → 256)

# Phase 17 결과 — summary 에서 추출
# 비교 mapping: p16 의 DST_RigL_50 ↔ Phase 17 의 DST_RigL_p16hp (같은 hp, scale 만 다름)
p17 = {name: summary[name][0] for name in [m["name"] for m in MODES]}

fig, ax = plt.subplots(1, 1, figsize=(11, 5))
modes_plot = ["dense", "static_prune_50", "DST_RigL (p16hp control)", "grown"]
x = list(range(len(modes_plot)))
p16_vals = [
    p16a_baselines["dense"],
    p16a_baselines["static_50"],
    p16a_baselines["DST_RigL_50"],
    p16b_grown,
]
# Phase 17 의 p16hp 가 Phase 16 와 같은 hp 이므로 직접 비교 (순수 scale 효과)
p17_vals = [
    p17["dense_large"],
    p17["static_prune_50"],
    p17["DST_RigL_p16hp"],  # ← p16hp control 만 비교
    p17["grown"],
]
width = 0.35
ax.bar(
    [i - width / 2 for i in x],
    p16_vals,
    width,
    label="Phase 16 (~1M, 1500 step)",
    color="lightblue",
)
ax.bar(
    [i + width / 2 for i in x], p17_vals, width, label="Phase 17 (~5M, 5000 step)", color="darkblue"
)
ax.set_xticks(x)
ax.set_xticklabels(modes_plot, rotation=15)
ax.set_ylabel("final loss (mean over 2 seeds)")
ax.set_title("Phase 17 vs Phase 16 — 순수 scale 효과 (같은 hp 로 비교)")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
fig.savefig(out_dir / "phase16_vs_phase17.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {out_dir / 'phase16_vs_phase17.png'}")
print(f"\nphase17 의 DST_RigL_p17hp (보수적 hp) = {p17['DST_RigL_p17hp']:.4f} (hp ablation 참고용)")

## 6. 결론 / 다음 단계

(셀 출력 보고 사용자가 채울 영역)

**핵심 질문 답변**:
- DST_RigL advantage 가 scale-up 으로 발현되었나?
- grown 의 capacity gap 이 축소되었나?
- Phase 16 대비 모든 mode 의 절대 loss 가 명확히 낮아짐?

**Phase 18 후보**:
- (advantage 발현 시) 16a + 16b 결합 — grow + DST 동시
- (여전히 미실현 시) dataset 확대 (OpenWebText subset) 또는 method 개선 (Net2WiderNet duplication / RigL ablation)
- (명백한 우수성 발견 시) scaling law 분석 + 결과 정리